# spinangle — gated spherical nGPT-JEPA vs. official LeWM (Colab GPU)

Baseline = **official LeWM, unchanged** (`lucas-maes/le-wm`). The big cell below clones the harness, installs the LeWM stack, reproduces LeWM (Phase 1), then trains/evals the nGPT-JEPA variants on its own benchmark + planner.

Set **Runtime → GPU** first. Start with `BENCH='tworoom'` (lightest data) and `EPOCHS=5` to validate the loop; then scale to `EPOCHS=100` and add variants.

Docs in the repo: `official_lewm_reproduction.md`, `RUN_MATRIX.md`, `report.md`.

## ▶️ The big cell (edit config, run)

In [ ]:
#@title 🌀 spinangle: gated spherical nGPT-JEPA vs official LeWM — one-shot runner
# Runtime > Change runtime type > GPU.  Edit the config block, then Run.
import os, subprocess, glob

# ----------------------------- config -----------------------------
BENCH    = "tworoom"   # tworoom (3.4G, lightest) | pusht (13G) | reacher (24G) | cube (46G)
EPOCHS   = 5           # 5 = quick validation; set 100 for the matched-compute comparison
VARIANTS = ["official_lewm", "gated_spherical"]   # also: lewm_nosigreg, simple_spherical,
#            fullish_residual, gated_spherical_projector_sigreg, gated_spherical_memory,
#            gated_spherical_ssm, ngpt_lr, ngpt_lr_groups   (see RUN_MATRIX.md)
GET_DATA = True        # download dataset (needed to train AND to sample eval episodes)
BRANCH   = "claude/upbeat-babbage-kbmgsr"
# ------------------------------------------------------------------

DATACFG = {"tworoom": "tworoom", "pusht": "pusht", "reacher": "dmc", "cube": "ogb"}[BENCH]
os.environ["STABLEWM_HOME"] = "/content/stable-wm"
os.environ["MUJOCO_GL"] = "egl"

def sh(cmd, check=False):
    print(f"\n\033[1;36m$ {cmd}\033[0m", flush=True)
    return subprocess.run(cmd, shell=True, check=check).returncode

# 0) GPU + clone + install -----------------------------------------------------
sh("nvidia-smi -L || echo '⚠️  NO GPU — Runtime > Change runtime type > GPU'")
if not os.path.isdir("/content/spinangle"):
    sh(f"git clone -b {BRANCH} https://github.com/turtlenottortoise/spinangle.git /content/spinangle")
else:
    sh("cd /content/spinangle && git pull")
os.chdir("/content/spinangle")
sh("pip -q install 'stable-worldmodel[train,env]' matplotlib einops huggingface_hub")

# 1) CPU smoke test — validates EVERY variant (no data/GPU needed) --------------
sh("python smoke_test.py && python metrics.py", check=True)

# 2) Official data + pretrained checkpoint -------------------------------------
sh(f"python scripts/download_assets.py --benchmark {BENCH} --ckpt" + (" --data" if GET_DATA else ""))

# 3) PHASE 1 — reproduce official LeWM from the pretrained checkpoint (the gate)
sh(f"python eval.py --config-name={BENCH}.yaml policy={BENCH}/lewm")

# 4) PHASES 2-3 — train + eval variants (same eval.py + planner; only +experiment)
CKPT = f"{os.environ['STABLEWM_HOME']}/checkpoints/{BENCH}"
for v in VARIANTS:
    sh(f"python train.py +experiment={v} data={DATACFG} "
       f"output_model_name={BENCH}/{v} trainer.max_epochs={EPOCHS} wandb.enabled=false")
    # keep only the newest epoch ckpt so load_pretrained finds a single .pt
    pts = sorted(glob.glob(f"{CKPT}/{v}/weights_epoch_*.pt"), key=os.path.getmtime)
    for old in pts[:-1]:
        os.remove(old)
    sh(f"python eval.py --config-name={BENCH}.yaml policy={BENCH}/{v} "
       f"|| echo '⚠️  trained-ckpt eval path — see RUN_MATRIX.md notes'")
    sph = "" if v in ("official_lewm", "lewm_nosigreg") else "--spherical"
    sh(f"python scripts/eval_latent_metrics.py --policy {BENCH}/{v} --data {DATACFG} "
       f"--benchmark {BENCH} --variant {v} {sph} --horizon 20 --num_batches 16 || true")

# 5) Plots ---------------------------------------------------------------------
sh("python scripts/make_plots.py")
from IPython.display import Image, display
for p in ["success_vs_steps", "rollout_error_vs_horizon", "retrieval_vs_steps",
          "rank_clumping", "planning_budget_curve"]:
    fp = f"/content/spinangle/plots/{p}.png"
    if os.path.exists(fp):
        display(Image(fp))
print("\n✅ DONE — results in results/all_runs.csv, plots in plots/. Fill report.md.")


## Phase 7 — νGPT scaling (optional)
Compare global LR vs. high LR vs. per-group LR (higher LR on the spherical transition nets). Run after the loop above for the same `BENCH`/`DATACFG`.

In [ ]:
BENCH, DATACFG, EPOCHS = 'tworoom', 'tworoom', 100
for v in ['gated_spherical', 'ngpt_lr', 'ngpt_lr_groups']:
    !python train.py +experiment={v} data={DATACFG} output_model_name={BENCH}/{v} \
        trainer.max_epochs={EPOCHS} wandb.enabled=false
    !python eval.py --config-name={BENCH}.yaml policy={BENCH}/{v} || true


## Persist results back to the branch (optional)

In [ ]:
!cd /content/spinangle && git add results/all_runs.csv plots/*.png && \
  git -c user.email=colab@local -c user.name=colab commit -m 'colab: results' && \
  git push || echo 'configure git auth (token) to push'
